## Feature Engineering
Builds a 214-feature-per-frame representation from raw normalised keypoints.

In [8]:
import numpy as np
import os

In [9]:
X = np.load(r"D:\CA_POC\new_processed\X.npy")   # (N, 100, 99)
y = np.load(r"D:\CA_POC\new_processed\y.npy")

print(X.shape, y.shape)

(976, 100, 99) (976,)


In [10]:
# ── Landmark index map ─────────────────────────────────────────────────────
MP = dict(
    L_SHOULDER=11, R_SHOULDER=12,
    L_ELBOW=13,    R_ELBOW=14,
    L_WRIST=15,    R_WRIST=16,
    L_HIP=23,      R_HIP=24,
    L_KNEE=25,     R_KNEE=26,
    L_ANKLE=27,    R_ANKLE=28,
    NOSE=0,
)

def reshape_kp(sequence):
    return sequence.reshape(-1, 33, 3)

def calculate_angle(a, b, c):
    """Interior angle at b in degrees."""
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))

In [11]:
# ── Base features ──────────────────────────────────────────────────────────

def extract_angles(sequence):
    """
    6 joint angles — extended from original 4 to include ankle angles.
    (T, 6): L-elbow, R-elbow, L-knee, R-knee, L-ankle, R-ankle
    """
    seq  = reshape_kp(sequence)
    out  = []

    for kp in seq:
        out.append([
            calculate_angle(kp[MP['L_SHOULDER']], kp[MP['L_ELBOW']], kp[MP['L_WRIST']]),   # L elbow
            calculate_angle(kp[MP['R_SHOULDER']], kp[MP['R_ELBOW']], kp[MP['R_WRIST']]),   # R elbow
            calculate_angle(kp[MP['L_HIP']],      kp[MP['L_KNEE']], kp[MP['L_ANKLE']]),    # L knee
            calculate_angle(kp[MP['R_HIP']],      kp[MP['R_KNEE']], kp[MP['R_ANKLE']]),    # R knee
            calculate_angle(kp[MP['L_KNEE']],     kp[MP['L_ANKLE']], kp[MP['L_ANKLE']] + np.array([0.1,0,0])),  # L ankle proxy
            calculate_angle(kp[MP['R_KNEE']],     kp[MP['R_ANKLE']], kp[MP['R_ANKLE']] + np.array([0.1,0,0])),  # R ankle proxy
        ])

    return np.array(out)   # (T, 6)


def extract_velocity(sequence):
    """First-order temporal derivative (frame-to-frame displacement). (T, 99)"""
    v = np.diff(sequence, axis=0)
    return np.vstack([v, v[-1]])   # pad last frame to keep T unchanged


def extract_acceleration(sequence):
    """Second-order temporal derivative. (T, 99)"""
    v = np.diff(sequence, axis=0)
    a = np.diff(v, axis=0)
    return np.vstack([a, a[-1], a[-1]])   # pad 2 frames

In [12]:
# ── Cricket-specific features ──────────────────────────────────────────────

def extract_cricket_features(sequence):
    """
    10 cricket biomechanical features per frame:
      1.  Bat angle         — wrist-to-elbow vector vs horizontal
      2.  Foot lead         — R-ankle x minus L-ankle x (normalised by hip width)
      3.  Body lean angle   — shoulder-to-hip vector vs horizontal
      4.  Head-over-knee    — horizontal distance nose to front knee
      5.  Wrist height      — R-wrist y relative to hip midpoint (normalised by torso)
      6.  Hip rotation      — angle of hip line vs frontal plane
      7.  Shoulder rotation — angle of shoulder line vs frontal plane
      8.  Knee bend (R)     — R-hip / R-knee / R-ankle angle (duplicate capture for emphasis)
      9.  Elbow gap         — distance between wrists (hand separation on bat handle proxy)
      10. Head tilt         — nose y relative to shoulder midpoint
    """
    seq  = reshape_kp(sequence)
    out  = []

    for kp in seq:
        # 1. Bat angle
        bat_vec   = kp[MP['R_WRIST']] - kp[MP['R_ELBOW']]
        bat_angle = np.degrees(np.arctan2(bat_vec[1], bat_vec[0]))

        # 2. Foot lead (normalised by hip width)
        hip_w     = abs(kp[MP['L_HIP'], 0] - kp[MP['R_HIP'], 0]) + 1e-6
        foot_lead = (kp[MP['R_ANKLE'], 0] - kp[MP['L_ANKLE'], 0]) / hip_w

        # 3. Body lean
        body_vec   = kp[MP['R_SHOULDER']] - kp[MP['R_HIP']]
        body_angle = np.degrees(np.arctan2(body_vec[1], body_vec[0]))

        # 4. Head-over-knee
        head_knee = abs(kp[MP['NOSE'], 0] - kp[MP['R_KNEE'], 0])

        # 5. Wrist height (normalised by torso length)
        hip_y   = (kp[MP['L_HIP'], 1] + kp[MP['R_HIP'], 1]) / 2
        sh_y    = (kp[MP['L_SHOULDER'], 1] + kp[MP['R_SHOULDER'], 1]) / 2
        torso   = abs(sh_y - hip_y) + 1e-6
        w_height = (hip_y - kp[MP['R_WRIST'], 1]) / torso

        # 6. Hip rotation
        hip_vec      = kp[MP['R_HIP']] - kp[MP['L_HIP']]
        hip_rot      = np.degrees(np.arctan2(hip_vec[2], hip_vec[0]))

        # 7. Shoulder rotation
        sh_vec       = kp[MP['R_SHOULDER']] - kp[MP['L_SHOULDER']]
        sh_rot       = np.degrees(np.arctan2(sh_vec[2], sh_vec[0]))

        # 8. Knee bend (R) — repeated intentionally for feature emphasis
        knee_bend = calculate_angle(
            kp[MP['R_HIP']], kp[MP['R_KNEE']], kp[MP['R_ANKLE']]
        )

        # 9. Elbow gap (wrist separation)
        elbow_gap = np.linalg.norm(kp[MP['L_WRIST']] - kp[MP['R_WRIST']])

        # 10. Head tilt
        head_tilt = kp[MP['NOSE'], 1] - (sh_y)

        out.append([
            bat_angle, foot_lead, body_angle, head_knee,
            w_height, hip_rot, sh_rot, knee_bend,
            elbow_gap, head_tilt,
        ])

    return np.array(out)   # (T, 10)

In [13]:
def build_features(sequence):
    """
    Final feature vector per timestep — 214 dims:
      99 (keypoints) + 99 (velocity) + 6 (angles) + 10 (cricket) = 214
    """
    features = np.concatenate([
        sequence,                       # 99
        extract_velocity(sequence),     # 99
        extract_angles(sequence),       #  6
        extract_cricket_features(sequence),  # 10
    ], axis=1)

    assert features.shape[1] == 214, f'Feature dim mismatch: {features.shape[1]}'
    return features

In [14]:
X_new = []

for seq in X:
    X_new.append(build_features(seq))

X_new = np.array(X_new)
print("New Shape:", X_new.shape)   # expect (N, 100, 214)

np.save(r"D:\CA_POC\new_processed\X_features.npy", X_new)
np.save(r"D:\CA_POC\new_processed\y.npy", y)

New Shape: (976, 100, 214)
